# Lesson 14A: Self-Supervised Learning - Theory

Learn representations without labels.

**Self-Supervised Learning** creates pseudo-labels from data structure.

**Contrastive Learning:**
- Pull similar samples (positive pairs) together
- Push dissimilar samples (negative pairs) apart

**Popular Methods:**
- **SimCLR**: Contrastive learning with data augmentation
- **MoCo**: Momentum Contrast with queue
- **CLIP**: Contrastive Language-Image Pre-training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

## SimCLR Algorithm

1. **Data Augmentation**: Apply two random augmentations to each image
   - x_i → x̃_i and x̃_j (positive pair)

2. **Encoder**: f(·) maps to representation space
   - h_i = f(x̃_i)

3. **Projection Head**: g(·) maps to contrastive space
   - z_i = g(h_i)

4. **Contrastive Loss (NT-Xent):**
   ```
   ℓ(i,j) = -log [exp(sim(z_i, z_j)/τ) / Σ_k exp(sim(z_i, z_k)/τ)]
   ```

**Key Insight:** Maximize agreement between different augmented views of same sample.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimCLR(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, projection_dim=64):
        super().__init__()
        # Encoder (simple MLP for demonstration)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection(h)
        return F.normalize(z, dim=-1)

def nt_xent_loss(z_i, z_j, temperature=0.5):
    """NT-Xent (Normalized Temperature-scaled Cross Entropy) loss"""
    batch_size = z_i.shape[0]
    
    # Compute similarity matrix
    z = torch.cat([z_i, z_j], dim=0)  # 2N x D
    sim_matrix = torch.mm(z, z.T) / temperature  # 2N x 2N
    
    # Mask out self-similarities
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    
    # Positive pairs: (i, j) and (j, i)
    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ])
    
    # NT-Xent loss
    loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
    return loss.mean()

# Test
model = SimCLR(input_dim=64, hidden_dim=128, projection_dim=64)

# Simulate two augmented views
x_i = torch.randn(32, 64)
x_j = torch.randn(32, 64)

z_i = model(x_i)
z_j = model(x_j)

loss = nt_xent_loss(z_i, z_j)
print(f"NT-Xent Loss: {loss.item():.4f}")
print(f"Projection shape: {z_i.shape}")
print("✅ SimCLR architecture and loss defined!")